In [0]:
project_identifier = "DAR065_1"
study_start_date = "2017-01-01"
study_end_date = "2026-06-30"

# Template IG policy: columns need both tags within these thresholds.
# Untagged columns are excluded unless explicitly reviewed below.
max_ig_risk = 3
max_ig_severity = 2

# Always excluded, even if source tags change.
columns_to_exclude = [
    "ADC_UPDT",
]

columns_to_include = {}
unfiltered_tables = []

# RDE tables: standard cohort-filtered project views.
rde_tables = [
    "rde_patient_demographics",
    "rde_all_diagnosis",
    "rde_all_problems",
    "rde_all_procedures",
    "rde_encounter",
    "rde_measurements",
    "rde_pharmacyorders",
    "rde_medadmin",
    "rde_radiology",
    "rde_blobdataset",  # approved anonymous text for MedCAT
]

# Bronze/map tables: standard cohort-filtered project views.
map_tables = [
    "map_person",
    "map_patient_journey",
]

omop_tables = []
mill_tables = []

# map_address, MediConnect and its MRN bridge are handled only in bespoke views below.
# Raw identifiers from those tables are never published.

core_facilities = [
    "RNJ BARTS",
    "5C4 BARTS",
    "RNJ ROYALLONDON",
    "R1H WHIPPSCROSS",
    "NUH NEWHAM",
    "RNJ NEWHAM",
    "RNJ MILE END",
    "5C4 MILE END",
    "RNJ LONDONCHEST",
    "BARTS AND THE LONDON NHS TRUST",
]

# OPCS codes used to identify new implants.
device_codes = [
    ("K601", "PPM", "PPM"),
    ("K605", "PPM", "PPM"),
    ("K606", "PPM", "PPM"),
    ("K608", "PPM", "PPM"),
    ("K609", "PPM", "PPM"),
    ("K611", "PPM", "PPM"),
    ("K615", "PPM", "PPM"),
    ("K616", "PPM", "PPM"),
    ("K618", "PPM", "PPM"),
    ("K619", "PPM", "PPM"),
    ("K701", "PPM", "LEADLESS_PPM"),
    ("K702", "PPM", "LEADLESS_PPM"),
    ("K607", "CRT", "CRT-P"),
    ("K617", "CRT", "CRT-P"),
    ("K591", "ICD", "ICD"),
    ("K592", "ICD", "ICD"),
    ("K598", "ICD", "ICD"),
    ("K599", "ICD", "ICD"),
    ("K721", "ICD", "S-ICD"),
    ("K596", "CRT", "CRT-D"),
]


def sql_quote(value):
    return "'" + str(value).replace("'", "''") + "'"


facility_sql = ", ".join(sql_quote(value) for value in core_facilities)
device_values_sql = ",\n      ".join(
    f"({sql_quote(code)}, {sql_quote(group)}, {sql_quote(subtype)})"
    for code, group, subtype in device_codes
)
encounter_date_sql = (
    "CAST(COALESCE(j.REG_DT_TM, j.ENCNTR_ARRIVE_DT_TM, "
    "j.INPATIENT_ADMIT_DT_TM, j.LOC_BEG_EFFECTIVE_DT_TM) AS DATE)"
)

## Cohort

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS 6_mgmt.cohorts")

cohort_sql = f"""
CREATE OR REPLACE VIEW 6_mgmt.cohorts.{project_identifier} AS
WITH device_code_map AS (
  SELECT *
  FROM VALUES
      {device_values_sql}
  AS d(OPCS4_CODE, DEVICE_GROUP, DEVICE_SUBTYPE)
),
adult_activity AS (
  SELECT
    CAST(j.PERSON_ID AS BIGINT) AS PERSON_ID,
    MAX(
      CASE
        WHEN lower(concat_ws(' ', j.MED_SERVICE_DESC, j.SPECIALTY_UNIT_DESC, j.NURSE_UNIT_DESC))
             RLIKE 'cardiology|interventional cardiology|electrophysiology|heart failure'
         AND lower(concat_ws(' ', j.MED_SERVICE_DESC, j.SPECIALTY_UNIT_DESC, j.NURSE_UNIT_DESC))
             NOT RLIKE 'paediatric|pediatric'
        THEN 1 ELSE 0
      END
    ) AS EVER_CARDIOLOGY_IND
  FROM 4_prod.bronze.map_patient_journey j
  INNER JOIN 4_prod.bronze.map_person p
    ON CAST(j.PERSON_ID AS BIGINT) = CAST(p.person_id AS BIGINT)
  WHERE upper(trim(j.FACILITY_DESC)) IN ({facility_sql})
    AND lower(coalesce(j.FINANCIAL_CLASS_DESC, '')) NOT LIKE '%private%'
    AND {encounter_date_sql} BETWEEN DATE '{study_start_date}' AND DATE '{study_end_date}'
    AND p.birth_datetime IS NOT NULL
    AND FLOOR(months_between({encounter_date_sql}, CAST(p.birth_datetime AS DATE)) / 12) >= 18
  GROUP BY CAST(j.PERSON_ID AS BIGINT)
),
implant_history AS (
  SELECT
    CAST(pr.PERSON_ID AS BIGINT) AS PERSON_ID,
    MIN(
      CASE
        WHEN CAST(pr.PROC_DT_TM AS DATE)
             BETWEEN DATE '{study_start_date}' AND DATE '{study_end_date}'
        THEN CAST(pr.PROC_DT_TM AS DATE)
      END
    ) AS FIRST_STUDY_IMPLANT_DATE,
    MIN(
      CASE
        WHEN CAST(pr.PROC_DT_TM AS DATE) < DATE '{study_start_date}'
        THEN CAST(pr.PROC_DT_TM AS DATE)
      END
    ) AS FIRST_PRIOR_IMPLANT_DATE
  FROM 4_prod.bronze.map_procedure pr
  INNER JOIN device_code_map d
    ON regexp_replace(upper(coalesce(pr.OPCS4_CODE, '')), '[^A-Z0-9]', '') = d.OPCS4_CODE
  WHERE CAST(pr.PROC_DT_TM AS DATE) <= DATE '{study_end_date}'
  GROUP BY CAST(pr.PERSON_ID AS BIGINT)
)
SELECT
  a.PERSON_ID,
  CASE
    WHEN i.FIRST_STUDY_IMPLANT_DATE IS NOT NULL THEN 'IMPLANTED_CIED'
    WHEN i.FIRST_PRIOR_IMPLANT_DATE IS NOT NULL THEN 'PRE_EXISTING_CIED'
    WHEN a.EVER_CARDIOLOGY_IND = 1 THEN 'NONIMPLANTED_CARDIOLOGY'
    ELSE 'NONIMPLANTED_OTHER_BARTS'
  END AS SUBCOHORT
FROM adult_activity a
LEFT JOIN implant_history i ON a.PERSON_ID = i.PERSON_ID
"""
spark.sql(cohort_sql)
print(f"Created cohort view: 6_mgmt.cohorts.{project_identifier}")

## Schema Setup

In [0]:
spark.sql("USE CATALOG 5_projects")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS 5_projects.{project_identifier}")

existing_views_df = spark.sql(f"SHOW VIEWS IN 5_projects.{project_identifier}")
existing_views = {row.viewName for row in existing_views_df.collect()}
for view_name in existing_views:
    spark.sql(f"DROP VIEW IF EXISTS 5_projects.{project_identifier}.{view_name}")
    print(f"Dropped view: 5_projects.{project_identifier}.{view_name}")

existing_tables_df = spark.sql(f"SHOW TABLES IN 5_projects.{project_identifier}")
for row in existing_tables_df.collect():
    if row.tableName not in existing_views and row.tableName != project_identifier:
        spark.sql(f"DROP TABLE IF EXISTS 5_projects.{project_identifier}.{row.tableName}")
        print(f"Dropped table: 5_projects.{project_identifier}.{row.tableName}")

## Helper Functions

In [0]:
def quote_identifier(value):
    tick = chr(96)
    return tick + value.replace(tick, tick + tick) + tick


def get_safe_columns(catalog, schema, table):
    """Return tagged columns within the template IG thresholds."""
    full_path = f"{catalog}.{schema}.{table}"
    all_columns = spark.table(full_path).columns

    safe_df = spark.sql(f"""
        SELECT r.column_name
        FROM (
            SELECT column_name, CAST(tag_value AS INT) AS risk_val
            FROM {catalog}.information_schema.column_tags
            WHERE schema_name = '{schema}'
              AND table_name = '{table}'
              AND tag_name = 'ig_risk'
        ) r
        JOIN (
            SELECT column_name, CAST(tag_value AS INT) AS sev_val
            FROM {catalog}.information_schema.column_tags
            WHERE schema_name = '{schema}'
              AND table_name = '{table}'
              AND tag_name = 'ig_severity'
        ) s ON r.column_name = s.column_name
        WHERE r.risk_val <= {max_ig_risk}
          AND s.sev_val <= {max_ig_severity}
    """)
    safe_columns = {row.column_name for row in safe_df.collect()}

    safe_columns |= set(columns_to_include.get(full_path, []))
    safe_columns |= set(columns_to_include.get("*", []))

    excluded_lower = {column.lower() for column in columns_to_exclude}
    return [
        column
        for column in all_columns
        if column in safe_columns and column.lower() not in excluded_lower
    ]


def find_person_id_column(catalog, schema, table):
    """Find the pseudonymous person identifier used to apply the cohort."""
    columns = spark.table(f"{catalog}.{schema}.{table}").columns
    candidates = [
        "PERSON_ID",
        "person_id",
        "Person_ID",
        "PERSONID",
        "PersonID",
        "participant_id",
    ]
    for candidate in candidates:
        if candidate in columns:
            return candidate
    for column in columns:
        if "person" in column.lower() and "id" in column.lower():
            return column
    return None


def create_cohort_filtered_view(catalog, schema, table, project_id, column_list=None, alias=None):
    """Create the standard template-style project view."""
    full_path = f"{catalog}.{schema}.{table}"
    view_name = alias or table

    if column_list is None:
        column_list = get_safe_columns(catalog, schema, table)
        if not column_list:
            print(f"WARNING: No columns passed IG filtering for {full_path}. Skipping.")
            return False

    person_id_column = find_person_id_column(catalog, schema, table)
    if column_list == ["*"]:
        columns_sql = "t.*"
    else:
        columns_sql = ", ".join(
            f"t.{quote_identifier(column)}" for column in column_list
        )

    if person_id_column:
        view_sql = f"""
        CREATE OR REPLACE VIEW 5_projects.{project_id}.{view_name} AS
        SELECT {columns_sql}
        FROM {full_path} t
        INNER JOIN 6_mgmt.cohorts.{project_id} c
          ON t.{quote_identifier(person_id_column)} = c.PERSON_ID
        """
    else:
        print(f"Note: no person ID column in {full_path}; view is not cohort-filtered.")
        view_sql = f"""
        CREATE OR REPLACE VIEW 5_projects.{project_id}.{view_name} AS
        SELECT {columns_sql}
        FROM {full_path} t
        """

    spark.sql(view_sql)
    print(f"Created view: 5_projects.{project_id}.{view_name}")
    return True

## Create Views

rde_blobdataset is included here so the project receives tagged anonymous text.
BlobContents and other direct/raw identifiers remain excluded.

In [0]:
created = []
failed = []

source_groups = [
    ("rde", rde_tables),
    ("bronze", map_tables),
    ("omop", omop_tables),
    ("raw", mill_tables),
]

for source_schema, tables in source_groups:
    for table in tables:
        try:
            if create_cohort_filtered_view(
                "4_prod", source_schema, table, project_identifier
            ):
                created.append(f"{source_schema}.{table}")
        except Exception as exc:
            failed.append((f"{source_schema}.{table}", str(exc)))
            print(f"ERROR: {source_schema}.{table}: {exc}")

for table in unfiltered_tables:
    try:
        catalog, schema, source_table = table.split(".")
        if create_cohort_filtered_view(
            catalog,
            schema,
            source_table,
            project_identifier,
            column_list=["*"],
        ):
            created.append(f"{schema}.{source_table}")
    except Exception as exc:
        failed.append((table, str(exc)))
        print(f"ERROR: {table}: {exc}")

## Bespoke Views

Only three side views are required:

1. device_summary — classified new implant procedures and age at implantation.
2. mediconnect_device — safely linked device hardware without MRN or serial number.
3. map_address_imd — IMD fields without postcode, address, UPRN or coordinates.

In [0]:
spark.sql(f"""
CREATE OR REPLACE VIEW 5_projects.{project_identifier}.device_summary AS
WITH device_code_map AS (
  SELECT *
  FROM VALUES
      {device_values_sql}
  AS d(OPCS4_CODE, DEVICE_GROUP, DEVICE_SUBTYPE)
)
SELECT
  CAST(pr.PERSON_ID AS BIGINT) AS PERSON_ID,
  CAST(pr.ENCNTR_ID AS BIGINT) AS ENCNTR_ID,
  CAST(pr.PROCEDURE_ID AS BIGINT) AS PROCEDURE_ID,
  CAST(pr.PROC_DT_TM AS DATE) AS IMPLANTATION_DATE,
  CAST(
    FLOOR(
      months_between(CAST(pr.PROC_DT_TM AS DATE), CAST(p.birth_datetime AS DATE)) / 12
    ) AS INT
  ) AS AGE_AT_IMPLANTATION,
  d.OPCS4_CODE,
  pr.OPCS4_TERM,
  d.DEVICE_GROUP,
  d.DEVICE_SUBTYPE
FROM 4_prod.bronze.map_procedure pr
INNER JOIN device_code_map d
  ON regexp_replace(upper(coalesce(pr.OPCS4_CODE, '')), '[^A-Z0-9]', '') = d.OPCS4_CODE
INNER JOIN 6_mgmt.cohorts.{project_identifier} c
  ON CAST(pr.PERSON_ID AS BIGINT) = c.PERSON_ID
LEFT JOIN 4_prod.bronze.map_person p
  ON CAST(pr.PERSON_ID AS BIGINT) = CAST(p.person_id AS BIGINT)
WHERE CAST(pr.PROC_DT_TM AS DATE)
      BETWEEN DATE '{study_start_date}' AND DATE '{study_end_date}'
""")
created.append("bespoke.device_summary")

In [0]:
spark.sql(f"""
CREATE OR REPLACE VIEW 5_projects.{project_identifier}.mediconnect_device AS
WITH alias_source AS (
  SELECT
    nullif(
      regexp_replace(
        regexp_replace(CAST(ALIAS AS STRING), '[^0-9]', ''),
        '^0+',
        ''
      ),
      ''
    ) AS MRN_NORM,
    CAST(PERSON_ID AS BIGINT) AS PERSON_ID
  FROM 4_prod.raw.mill_person_alias
  WHERE ACTIVE_IND = 1
    AND PERSON_ALIAS_TYPE_CD = 10
    AND (BEG_EFFECTIVE_DT_TM IS NULL OR BEG_EFFECTIVE_DT_TM <= current_timestamp())
    AND (END_EFFECTIVE_DT_TM IS NULL OR END_EFFECTIVE_DT_TM > current_timestamp())
),
unique_alias AS (
  SELECT MRN_NORM, MAX(PERSON_ID) AS PERSON_ID
  FROM alias_source
  WHERE MRN_NORM IS NOT NULL
  GROUP BY MRN_NORM
  HAVING COUNT(DISTINCT PERSON_ID) = 1
),
device_source AS (
  SELECT
    nullif(
      regexp_replace(
        regexp_replace(CAST(PATIENTID AS STRING), '[^0-9]', ''),
        '^0+',
        ''
      ),
      ''
    ) AS MRN_NORM,
    TYPE,
    MODEL_NO,
    MODEL_NAME,
    MANUFACTURER,
    IMPLANTED_DATE,
    LOCATION1,
    STATUS,
    DEVICE_TYPE,
    NO_OF_CHAMBERS
  FROM 4_prod.raw.mediconnect_mc_devices
)
SELECT
  a.PERSON_ID,
  d.DEVICE_TYPE,
  CASE
    WHEN d.DEVICE_TYPE IN (1, 2, 3, 4) THEN 'GENERATOR'
    WHEN d.DEVICE_TYPE IN (5, 6, 8, 9, 10, 11, 12) THEN 'LEAD'
    WHEN d.DEVICE_TYPE = 7 THEN 'MONITOR'
    ELSE 'OTHER'
  END AS DEVICE_ROLE,
  CASE
    WHEN d.DEVICE_TYPE = 1 THEN 'PPM'
    WHEN d.DEVICE_TYPE = 2 THEN 'CRT'
    WHEN d.DEVICE_TYPE = 3 THEN 'ICD'
    WHEN d.DEVICE_TYPE = 4 THEN 'CRT'
  END AS DEVICE_GROUP,
  CASE
    WHEN d.DEVICE_TYPE = 1 THEN 'PPM'
    WHEN d.DEVICE_TYPE = 2 THEN 'CRT-P'
    WHEN d.DEVICE_TYPE = 3 THEN 'ICD'
    WHEN d.DEVICE_TYPE = 4 THEN 'CRT-D'
    WHEN d.DEVICE_TYPE = 7 THEN 'ICM'
    ELSE d.TYPE
  END AS DEVICE_SUBTYPE,
  d.TYPE AS SOURCE_DEVICE_TYPE,
  d.MANUFACTURER,
  d.MODEL_NO,
  d.MODEL_NAME,
  CASE
    WHEN d.IMPLANTED_DATE IS NOT NULL
     AND year(d.IMPLANTED_DATE) > 1901
     AND CAST(d.IMPLANTED_DATE AS DATE) <= DATE '{study_end_date}'
    THEN CAST(d.IMPLANTED_DATE AS DATE)
  END AS IMPLANTATION_DATE,
  d.LOCATION1 AS DEVICE_LOCATION,
  d.STATUS AS SOURCE_STATUS,
  d.NO_OF_CHAMBERS
FROM device_source d
INNER JOIN unique_alias a ON d.MRN_NORM = a.MRN_NORM
INNER JOIN 6_mgmt.cohorts.{project_identifier} c ON a.PERSON_ID = c.PERSON_ID
""")
created.append("bespoke.mediconnect_device")

In [0]:
spark.sql(f"""
CREATE OR REPLACE VIEW 5_projects.{project_identifier}.map_address_imd AS
SELECT
  CAST(a.PARENT_ENTITY_ID AS BIGINT) AS PERSON_ID,
  a.ADDRESS_ID,
  try_cast(a.IMD_Decile AS INT) AS IMD_DECILE,
  try_cast(a.IMD_Quintile AS INT) AS IMD_QUINTILE,
  a.BEG_EFFECTIVE_DT_TM,
  a.END_EFFECTIVE_DT_TM,
  a.ACTIVE_IND,
  a.match_algorithm,
  a.match_confidence,
  a.match_quality
FROM 4_prod.bronze.map_address a
INNER JOIN 6_mgmt.cohorts.{project_identifier} c
  ON CAST(a.PARENT_ENTITY_ID AS BIGINT) = c.PERSON_ID
WHERE upper(a.PARENT_ENTITY_NAME) = 'PERSON'
""")
created.append("bespoke.map_address_imd")

## Schema Documentation View

In [0]:
spark.sql(f"""
CREATE OR REPLACE VIEW 5_projects.{project_identifier}.schema AS
SELECT
  table_name,
  column_name,
  COALESCE(comment, '') AS column_comment
FROM 5_projects.information_schema.columns
WHERE table_catalog = '5_projects'
  AND lower(table_schema) = lower('{project_identifier}')
  AND table_name != 'schema'
ORDER BY table_name, column_name
""")
print(f"Created schema view: 5_projects.{project_identifier}.schema")

## Summary

In [0]:
print("=" * 60)
print(f"Project setup complete: {project_identifier}")
print("=" * 60)
print(f"Cohort: 6_mgmt.cohorts.{project_identifier}")
print(f"Schema: 5_projects.{project_identifier}")
print(f"IG thresholds: risk <= {max_ig_risk}, severity <= {max_ig_severity}")
print("Untagged columns are excluded.")
print(f"Views created: {len(created)}")
for view_name in created:
    print(f"  + {view_name}")
if failed:
    print(f"Failed: {len(failed)}")
    for table_name, error in failed:
        print(f"  x {table_name}: {error}")